In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Copy the zip from Drive to local fast storage and unzip
# (Assuming you uploaded it to the root of your Google Drive)
!cp "/content/drive/MyDrive/vaidya_slm.zip" "/content/vaidya_slm.zip"
!unzip -q /content/vaidya_slm.zip -d /content/vaidya_slm

print("Successfully unzipped to /content/vaidya_slm")


Mounted at /content/drive
Successfully unzipped to /content/vaidya_slm


In [ ]:
# We install PyPDF2 for scraping the medical books along with LLM dependencies
!pip install -q -U peft transformers datasets trl accelerate bitsandbytes PyPDF2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.8 MB/s eta 0:00:00


In [ ]:
from google.colab import files
import os

print("Please upload your kaggle.json file")
files.upload()

# Safely move it to the required hidden folder
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


Please upload your kaggle.json file


Saving kaggle.json to kaggle.json


In [ ]:
%%writefile /content/vaidya_slm/vaidya_slm/scripts/data/download_datasets.py
import os, subprocess
from pathlib import Path

def main():
    data_dir = Path("/content/vaidya_slm/vaidya_slm/data/raw")
    data_dir.mkdir(parents=True, exist_ok=True)

    # 1. Archive.org (Using wget to bypass colab requests block)
    datasets = {
        "charaka_samhita.pdf": "https://archive.org/download/CharakaSamhitaVol1/CharakaSamhitaVol1_text.pdf",
        "sushruta_samhita.pdf": "https://archive.org/download/sushrutasamhita00kunjgoog/sushrutasamhita00kunjgoog_bw.pdf",
        "ashtanga_hridayam.pdf": "https://archive.org/download/AsthangaHridayamSanskritEnglsih/AsthangaHridayam%20Sanskrit%20%20Englsih.pdf"
    }

    for filename, url in datasets.items():
        output_file = data_dir / filename
        if not output_file.exists():
            print(f"Downloading {filename} via wget...")
            subprocess.run(["wget", "-q", "--show-progress", "-U", "Mozilla/5.0", "-O", str(output_file), url])

    # 2. Kaggle dataset
    print("\nDownloading Kaggle Ayurvedic Herbs...")
    subprocess.run(["kaggle", "datasets", "download", "-d", "harshavardhan21/ayurvedic-herbs", "-p", str(data_dir), "--unzip"])

if __name__ == "__main__":
    main()


Overwriting /content/vaidya_slm/vaidya_slm/scripts/data/download_datasets.py


In [ ]:
%%writefile /content/vaidya_slm/vaidya_slm/scripts/data/preprocess_ayurvedic.py
import json, os, re, pandas as pd
from datasets import load_dataset
from pathlib import Path

def clean_ocr_noise(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\xff]*', '', text)
    return re.sub(r'[^\w\s\.,;?\-()\[\]]', '', re.sub(r'\s+', ' ', text)).strip()

def process_pdf(pdf_path):
    text_corpus = ""
    try:
        import PyPDF2
        with open(pdf_path, 'rb') as file:
            for page in PyPDF2.PdfReader(file).pages:
                extracted = page.extract_text()
                if extracted: text_corpus += clean_ocr_noise(extracted) + "\n"
    except Exception as e:
        print(f"Error processing {pdf_path}: {e}")
    return text_corpus

def map_text_to_instructions(text_corpus, source_name):
    records = []
    sentences = [clean_ocr_noise(s.strip()) for s in text_corpus.split('.') if len(s.strip()) > 10]
    for i, sentence in enumerate(sentences):
        sentence_lower = sentence.lower()
        if any(dosha in sentence_lower for dosha in ['vata', 'pitta', 'kapha']):
            target_dosha = "Vata" if 'vata' in sentence_lower else ("Pitta" if 'pitta' in sentence_lower else "Kapha")
            remedy_context = " ".join(sentences[i:i+2]) if i+2 < len(sentences) else sentence
            records.append({
                "instruction": f"Based on {source_name}, analyze the symptom and suggest a remedy for the dosha imbalance.",
                "input": sentence,
                "output": {"dosha": target_dosha, "reason": f"Symptom correlated with {target_dosha} aggravation in {source_name}.", "remedy": remedy_context}
            })
    return list({v['input']:v for v in records}.values())

def build_instruction_dataset():
    data_dir = Path("/content/vaidya_slm/vaidya_slm/data/raw")
    output_path = Path("/content/vaidya_slm/vaidya_slm/data/processed/ayurvedic_dataset.json")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    all_records = []

    # Process PDFs
    for pdf_file in ['charaka_samhita.pdf', 'sushruta_samhita.pdf', 'ashtanga_hridayam.pdf']:
        path = data_dir / pdf_file
        if path.exists():
            print(f"Processing Text: {pdf_file}")
            all_records.extend(map_text_to_instructions(process_pdf(str(path)), pdf_file))

    # Process Kaggle CSV
    for csv_file in data_dir.glob("*.csv"):
        print(f"Parsing CSV Herbs map: {csv_file.name}")
        df = pd.read_csv(str(csv_file))
        for _, row in df.iterrows():
            if len(row.values) > 1:
                all_records.append({"instruction": "Suggest an Ayurvedic herb given the symptom.", "input": clean_ocr_noise(str(row.values[0])), "output": {"dosha": "Unknown - General Remedy", "reason": "Ayurvedic Herbs Database.", "remedy": clean_ocr_noise(str(row.values[1]))}})

    # Process IndicGLUE
    print("Fetching IndicGLUE via HuggingFace...")
    try:
        indic_dataset = load_dataset("indic_glue", "copa.hi", split="train[:500]") # Fixed from sna.hi to copa.hi
        for item in indic_dataset:
            all_records.append({"instruction": "Translate and classify.", "input": item['premise'], "output": {"dosha": "N/A", "reason": "Language Translation Check", "remedy": "N/A"}})
    except Exception as e: print(f"IndicGLUE skipped: {e}")

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(all_records, f, indent=4, ensure_ascii=False)
    print(f"\nGenerated {len(all_records)} knowledge pairs. Ready for fine-tuning!")

if __name__ == "__main__":
    build_instruction_dataset()


Overwriting /content/vaidya_slm/vaidya_slm/scripts/data/preprocess_ayurvedic.py


In [ ]:
import json, os
from datasets import load_dataset

os.makedirs("/content/vaidya_slm/vaidya_slm/data/processed", exist_ok=True)
output_path = "/content/vaidya_slm/vaidya_slm/data/processed/ayurvedic_dataset.json"

print("Downloading Medical/Remedies dataset from HuggingFace...")

# Loading an open-source medical/remedies QA dataset
# (Using 'medalpaca/medical_meadow_health_advice' as a reliable stand-in for hackathons)
fallback_dataset = load_dataset("medalpaca/medical_meadow_health_advice", split="train[:500]")

all_records = []
for item in fallback_dataset:
    all_records.append({
        "instruction": "Analyze the following symptom and suggest a remedy.",
        "input": item.get('input', item.get('instruction', '')),
        "output": {
            "dosha": "Unknown - General Checkup",
            "reason": "Derived from Alternative Medical Dataset",
            "remedy": item.get('output', 'Consult a Vaidya/Doctor.')
        }
    })

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_records, f, indent=4, ensure_ascii=False)

print(f"Successfully generated {len(all_records)} knowledge pairs from HuggingFace!")
print("You can now proceed directly to Cell 4 for QLoRA Fine-tuning!")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

medical_meadow_health_advice.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/8676 [00:00<?, ? examples/s]

Successfully generated 500 knowledge pairs from HuggingFace!
You can now proceed directly to Cell 4 for QLoRA Fine-tuning!


In [ ]:
%cd /content/vaidya_slm/vaidya_slm
!python scripts/data/download_datasets.py
!python scripts/data/preprocess_ayurvedic.py


/content/vaidya_slm/vaidya_slm

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata
Processing Text: charaka_samhita.pdf
Error processing /content/vaidya_slm/vaidya_slm/data/raw/charaka_samhita.pdf: Cannot read an empty file
Processing Text: sushruta_samhita.pdf
Error processing /content/vaidya_slm/vaidya_slm/data/raw/sushruta_samhita.pdf: Cannot read an empty file
Processing Text: ashtanga_hridayam.pdf
Error processing /content/vaidya_slm/vaidya_slm/data/raw/ashtanga_hridayam.pdf: Cannot read an empty file
Fetching IndicGLUE via HuggingFace...
copa.hi/train-00000-of-00001.parquet: 100% 41.8k/41.8k [00:01<00:00, 32.1kB/s]
copa.hi/validation-00000-of-00001.parque(…): 100% 13.9k/13.9k [00:00<00:00, 33.9kB/s]
copa.hi/test-00000-of-00001.parquet: 100% 48.6k/48.6k [00:00<00:00, 60.0kB/s]
Generating train split: 100% 362/362 [00:00<00:00, 103619.60 examples/s]
Generating validation split: 100% 88/88 [00:00<00:00, 46621.04 examples/s]
G

In [ ]:
import trl, transformers, peft
print("trl:", trl.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)


trl: 0.29.1
transformers: 5.3.0
peft: 0.18.1


In [ ]:
import inspect
from trl import SFTTrainer
print(inspect.signature(SFTTrainer.__init__))


(self, model: 'str | PreTrainedModel | PeftModel', args: trl.trainer.sft_config.SFTConfig | transformers.training_args.TrainingArguments | None = None, data_collator: collections.abc.Callable[[list[typing.Any]], dict[str, typing.Any]] | None = None, train_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | None = None, eval_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | dict[str, datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset] | None = None, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.processing_utils.ProcessorMixin | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None, callbacks: list[transformers.trainer_callback.TrainerCallback] | None = None, optimizers: tuple[torch.optim.optimizer.Optimizer | None, torch.opt

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer
import os

# 1. Load Tokenizer
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 2. 4-bit Quantization — compute dtype changed to bfloat16 to match model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16  # CHANGED from float16 to bfloat16
)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

# 3. LoRA Config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 4. Load Dataset
raw_dataset = load_dataset(
    "json",
    data_files="/content/vaidya_slm/vaidya_slm/data/processed/ayurvedic_dataset.json",
    split="train"
)

# 5. Formatting function
def format_instruction(sample):
    return f"### Instruction:\n{sample['instruction']}\n\n### Input:\n{sample['input']}\n\n### Response:\nDosha: {sample['output']['dosha']}\nReason: {sample['output']['reason']}\nRemedy: {sample['output']['remedy']}"

# 6. Training Args — bf16 instead of fp16 to match TinyLlama's native precision
training_args = TrainingArguments(
    output_dir="/content/vaidya_slm/vaidya_slm/scripts/training/vaidya-slm-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    fp16=False,       # DISABLED
    bf16=True,        # ENABLED — matches TinyLlama's BFloat16 tensors
    max_steps=200,
    report_to="none"
)

# 7. SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=raw_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    formatting_func=format_instruction,
)

print("Starting AI Training...")
trainer.train()

# 8. Save to Google Drive
drive_backup = "/content/drive/MyDrive/vaidya-slm-lora-final"
os.makedirs(drive_backup, exist_ok=True)
trainer.model.save_pretrained(drive_backup)
tokenizer.save_pretrained(drive_backup)
print(f"Training Complete! Saved to {drive_backup}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting AI Training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,3.003522
20,2.047291
30,1.337374
40,1.230362
50,1.191452
60,1.140755
70,1.085995
80,1.067169
90,1.061562
100,1.113257


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Training Complete! Saved to /content/drive/MyDrive/vaidya-slm-lora-final


In [ ]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Erase memory so Colab doesn't crash during the merge
torch.cuda.empty_cache()
gc.collect()

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = "/content/drive/MyDrive/vaidya-slm-lora-final"
save_path = "/content/drive/MyDrive/vaidya-slm-merged"

print("1. Loading base model into CPU...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="cpu"
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

print("2. Fusing the Medical Adapters...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model = model.merge_and_unload()

print("3. Saving unified Offline Edge model...")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print("Merge finished successfully!")


`torch_dtype` is deprecated! Use `dtype` instead!


1. Loading base model into CPU...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2. Fusing the Medical Adapters...
3. Saving unified Offline Edge model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merge finished successfully!


In [ ]:
!pip install -q torch==2.10.0+cu128 torchvision==0.25.0+cu128 torchaudio==2.10.0+cu128 --index-url https://download.pytorch.org/whl/cu128


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 916.9/916.9 MB 462.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 75.2 MB/s eta 0:00:00


In [ ]:
!pip install -q torch==2.10.0+cu128 torchvision==0.25.0+cu128 torchaudio==2.10.0+cu128 --index-url https://download.pytorch.org/whl/cu128


In [ ]:
# Stage 1: Convert merged model to GGUF in f16 (DO NOT use q4_0 as outtype)
!python llama.cpp/convert_hf_to_gguf.py /content/drive/MyDrive/vaidya-slm-merged \
    --outfile /content/drive/MyDrive/vaidya-slm-f16.gguf \
    --outtype f16

# Stage 2: Compile the quantize binary from llama.cpp
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release && cmake --build build -j4

# Stage 3: Use the binary to quantize f16 → Q4_0 (INT4)
!/content/vaidya_slm/vaidya_slm/llama.cpp/build/bin/llama-quantize \
    /content/drive/MyDrive/vaidya-slm-f16.gguf \
    /content/drive/MyDrive/vaidya-slm-1.1b-instruct-q4_0.gguf \
    Q4_0

print("Done! Your INT4 edge model is saved to Google Drive!")


INFO:hf-to-gguf:Loading model: vaidya-slm-merged
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_outp

In [ ]:
%cd /content/vaidya_slm/vaidya_slm
!git clone https://github.com/ggerganov/llama.cpp.git
!pip install -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py /content/drive/MyDrive/vaidya-slm-merged --outfile /content/drive/MyDrive/vaidya-slm-1.1b-instruct.gguf --outtype q4_0

print("All done! Your finished hackathon model is waiting in Google Drive as 'vaidya-slm-1.1b-instruct.gguf'")


/content/vaidya_slm/vaidya_slm
Cloning into 'llama.cpp'...
remote: Enumerating objects: 85315, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 85315 (delta 13), reused 9 (delta 9), pack-reused 85274 (from 2)
Receiving objects: 100% (85315/85315), 343.62 MiB | 23.95 MiB/s, done.
Resolving deltas: 100% (61602/61602), done.
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00


usage: convert_hf_to_gguf.py [-h] [--vocab-only] [--outfile OUTFILE]
                             [--outtype {f32,f16,bf16,q8_0,tq1_0,tq2_0,auto}]
                             [--bigendian] [--use-temp-file] [--no-lazy]
                             [--model-name MODEL_NAME] [--verbose]
                             [--split-max-tensors SPLIT_MAX_TENSORS]
                             [--split-max-size SPLIT_MAX_SIZE] [--dry-run]
                             [--no-tensor-first-split] [--metadata METADATA]
                             [--print-supported-models] [--remote] [--mmproj]
                             [--mistral-format]
                             [--disable-mistral-community-chat-template]
                             [--sentence-transformers-dense-modules]
                             [--fuse-gate-up-exps]
                             [model]
convert_hf_to_gguf.py: error: argument --outtype: invalid choice: 'q4_0' (choose from f32, f16, bf16, q8_0, tq1_0, tq2_0, auto)
All done

In [ ]:
from google.colab import files
import shutil

# Zip all your trained model files into one archive
shutil.make_archive("/content/vaidya_slm_trained_models", "zip", "/content/drive/MyDrive", ".")

# Download the zip to your local PC
files.download("/content/vaidya_slm_trained_models.zip")


OSError: [Errno 95] Operation not supported: '/content/drive/MyDrive/Joshitha_resume.gdoc'

In [ ]:
from google.colab import files
import shutil, os

# Zip only the specific Vaidya SLM files — NOT the whole drive root
os.makedirs("/content/vaidya_export", exist_ok=True)

# Copy only the project files we need
shutil.copy("/content/drive/MyDrive/vaidya-slm-1.1b-q4_0.gguf", "/content/vaidya_export/")
shutil.copytree("/content/drive/MyDrive/vaidya-slm-lora-final", "/content/vaidya_export/vaidya-slm-lora-final")

# Now zip the folder safely
shutil.make_archive("/content/vaidya_slm_models", "zip", "/content/vaidya_export")

# Download!
files.download("/content/vaidya_slm_models.zip")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/vaidya-slm-1.1b-q4_0.gguf'